# Exploratory statistics and aircraft profiles

Spearman analysis is descriptive. Freeze the date-selection rule before inspecting coefficients/p-values. Whole-period k-means is descriptive and is not fed into predictive models. The main inferential test of the research question is the OLS interaction model in the next section (CC-8); Spearman below is secondary.

In [ ]:
from contextlib import closing
from pathlib import Path
import sqlite3
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
DB = ROOT / "data/processed/aircraft.sqlite"
if not DB.exists():
    raise RuntimeError("Collect real data and run src/prepare.py first.")
with closing(sqlite3.connect(DB)) as db:
    data = pd.read_sql_query("SELECT * FROM observations", db)
    airports = pd.read_sql_query("SELECT * FROM airports", db)
    audit = pd.read_sql_query("SELECT * FROM integration_audit", db)
data["observed_at_utc"] = pd.to_datetime(data.observed_at_utc, utc=True)
OUT = ROOT / "outputs"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


## Research question, part 1: OLS with altitude × zone interaction (CC-8)

`src/inference.py` fits, on all observations (descriptive inference, not prediction):

`speed_kmh ~ (alt_low_km + alt_high_km) * C(zone, Treatment('far')) + C(airport) + C(flight_phase)`

- `alt_low_km` / `alt_high_km`: altitude in km below / above the knot (`altitude_knot_m`, 3,000 m), so every slope is in **km/h per 1,000 m**.
- Standard errors are **clustered by aircraft** (`icao24`): many positions of one aircraft are not independent observations. Inference uses t with (aircraft − 1) degrees of freedom.
- **Hypothesis 1** (slide 11): `alt_low_km:C(zone, ...)[T.near]` and `alt_high_km:C(zone, ...)[T.near]` > 0 with p < 0.05; the joint Wald test «altitude x zone: near vs far» tests both at once.
- Variants: (a) near zone only, ZRH vs GVA vs BSL; (b) aircraft category × altitude with the same controls. Levels with fewer than `inference.min_aircraft_per_level` aircraft are left out (see `*_levels` tables).
- The residual profile checks the knot: a systematic arc above ~10 km would suggest a second knot (`altitude_knot2_m`, proposal only, decided by the group).

Refuses to run on fewer than seven observed UTC dates.

In [ ]:
import sys
from IPython.display import Image
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import inference
config = json.loads((ROOT / "config.json").read_text())
with closing(sqlite3.connect(DB)) as db:
    enriched = pd.read_sql_query("SELECT * FROM observations_enriched", db)
print("Main model:", inference.formulas(config)["main"])
ols = inference.run_inference(enriched, config)  # 7-date guard inside
for name, table in ols.items():
    table.to_csv(OUT / f"tables/ols_{name}.csv", index=False)
display(ols["interaction_tests"].round(4))
display(ols["main_coefficients"][["term", "estimate_kmh", "ci_low_kmh", "ci_high_kmh", "p_value",
                                  "estimate_ms", "n_observations", "n_aircraft"]].round(3))
display(ols["zone_slopes"][["zone", "segment", "slope_kmh_per_1000m", "ci_low_kmh", "ci_high_kmh",
                            "slope_ms_per_1000m", "n_observations", "n_aircraft"]].round(2))
inference.slope_figure(ols["zone_slopes"], OUT / "figures/ols_slopes_by_zone.png", config)
display(Image(OUT / "figures/ols_slopes_by_zone.png"))
for name in ("airport_slopes", "category_slopes", "airport_levels", "category_levels"):
    if name in ols:
        print(name); display(ols[name].round(2))
inference.residual_figure(ols["residual_profile"], OUT / "figures/ols_residuals_by_altitude.png")
display(Image(OUT / "figures/ols_residuals_by_altitude.png"))

### Interpretation (group): altitude × zone

*Placeholder.* Report the near-vs-far interaction in km/h per 1,000 m with its 95 % CI and p-value (hypothesis 1), and the slopes per zone below and above 3,000 m. Say what the number does **not** show: it is an association across all observed positions, not a causal effect of the airport, and it depends on the aircraft mix per zone. Does the residual profile show an arc (plateau near cruise level)? If so, discuss the second knot with the group before changing the model.

In [ ]:
from scipy.stats import spearmanr
# Provisional deterministic rule: second observed UTC date. Confirm/freeze after pilot.
# One earliest position per aircraft on that date, reducing repeated-aircraft dependence.
dates = sorted(data.date_utc.unique())
if len(dates)<2:
    raise RuntimeError("Collect at least two dates before this exploratory analysis.")
analysis_day = dates[1]
one = data[data.date_utc.eq(analysis_day)].sort_values("observed_at_utc").drop_duplicates("icao24")
rows=[]
for zone in ["near","far"]:
    subset = one[one.zone.eq(zone)]
    if len(subset)<20 or subset.altitude_m.nunique()<2 or subset.velocity.nunique()<2:
        print(zone, "insufficient sample/variation; no test reported")
        continue
    result = spearmanr(subset.altitude_m,subset.velocity)
    rows.append({"analysis_day_utc":analysis_day,"zone":zone,"n_aircraft":len(subset),
        "spearman_rho":result.statistic,"p_value":result.pvalue})
statistics = pd.DataFrame(rows)
display(statistics)
statistics.to_csv(OUT / "tables/exploratory_correlations.csv",index=False)
# P-values are not probabilities that a hypothesis is true, or that chance caused the data.
# Separate p-values do not test whether near/far relationships differ.
# Discuss remaining shared-weather/time dependence and aircraft mix.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
profile = data.groupby("icao24")[["altitude_m","velocity"]].median().dropna()
if len(profile)<20:
    raise RuntimeError("Need at least 20 distinct aircraft profiles for clustering diagnostics.")
scaler = StandardScaler()
scaled = scaler.fit_transform(profile)
if np.unique(scaled,axis=0).shape[0]<6:
    raise RuntimeError("Too few distinct profiles to compare k=2..5.")
rows=[];fitted={}
for k in range(2,6):
    model = KMeans(n_clusters=k,n_init=10,random_state=42).fit(scaled)
    score = silhouette_score(scaled,model.labels_,sample_size=min(5000,len(scaled)),random_state=42)
    rows.append({"k":k,"inertia":model.inertia_,"silhouette":score})
    fitted[k]=model
scores=pd.DataFrame(rows)
display(scores)
k=int(scores.loc[scores.silhouette.idxmax(),"k"])
model=fitted[k]
profile["cluster"]=model.labels_
centroids = pd.DataFrame(scaler.inverse_transform(model.cluster_centers_),columns=["median_altitude_m","median_speed_m_s"])
display(centroids)
scores.to_csv(OUT / "tables/cluster_diagnostics.csv",index=False)
centroids.to_csv(OUT / "tables/cluster_centroids.csv",index_label="cluster")
sns.scatterplot(data=profile,x="altitude_m",y="velocity",hue="cluster",palette="tab10")
plt.xlabel("Aircraft median altitude (m)");plt.ylabel("Aircraft median ground speed (m/s)")
plt.tight_layout();plt.savefig(OUT / "figures/aircraft_profiles.png",dpi=160);plt.show()

## Group interpretation required

Interpret centroid values and sample sizes. Aircraft median profiles depend on where/when the aircraft was observed. They are not verified aircraft types. Compare silhouette and inertia rather than blindly choosing a pretty plot. Do not add a chi-squared test of speed bins versus these speed-derived clusters as independent evidence. The direct near/far adjusted relationship comparison requires additional common-support/interaction analysis.